<a href="https://colab.research.google.com/github/GitAnvi/hcl-project/blob/main/project_hcl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
from google.colab import files

# ==========================================
# STUDENT PERFORMANCE AND ATTENDANCE TRACKER
# ==========================================



print("Upload Student Performance Dataset")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded!")

file_name = list(uploaded.keys())[0]

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name, dtype={"Student_ID": str})

elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name, dtype={"Student_ID": str})

else:
    raise ValueError("Please upload a CSV or Excel file!")

subjects = [
    "Mathematics_Marks",
    "Physics_Marks",
    "Chemistry_Marks"
]

required_columns = [
    "Student_ID",
    "Student_Name",
    "Mathematics_Marks",
    "Physics_Marks",
    "Chemistry_Marks",
    "Attendance_%"
]

df.columns = df.columns.str.strip()

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")
df["Student_ID"] = df["Student_ID"].astype("string").str.strip()
df["Student_Name"] = df["Student_Name"].astype("string").str.strip()

if df[["Student_ID", "Student_Name"]].isna().any().any():
    raise ValueError("Student ID or Student Name is missing!")

if df[["Student_ID", "Student_Name"]].eq("").any().any():
    raise ValueError("Student ID or Student Name cannot be blank!")

if df["Student_ID"].duplicated().any():
    raise ValueError("Duplicate Student IDs found!")
for col in subjects + ["Attendance_%"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if df[subjects + ["Attendance_%"]].isna().any().any():
    raise ValueError("Missing or invalid marks/attendance values found!")
# Validate marks and attendance
for col in subjects + ["Attendance_%"]:
    if not df[col].between(0, 100).all():
        raise ValueError(f"{col} must be between 0 and 100!")

# STEP 4: Calculate Total Marks

df["Total_Marks"] = df[subjects].sum(axis=1)

# STEP 5: Calculate Percentage

df["Percentage"] = (
    df["Total_Marks"] / 300
) * 100

df["Percentage"] = df["Percentage"].round(2)

# STEP 6: Find Best Subject

def best_subject(row):

    highest = row[subjects].max()

    best = [
        s.replace("_Marks", "")
        for s in subjects
        if row[s] == highest
    ]

    return ", ".join(best)


df["Best_Subject"] = df.apply(best_subject, axis=1)

# STEP 7: Find Subject Needing Improvement

def improvement_subject(row):

    lowest = row[subjects].min()

    lowest_subject = [
        s.replace("_Marks", "")
        for s in subjects
        if row[s] == lowest
    ]

    return ", ".join(lowest_subject)


df["Needs_Improvement"] = df.apply(
    improvement_subject,
    axis=1
)

# STEP 8: Calculate Class Rank

df["Class_Rank"] = (
    df["Total_Marks"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# STEP 9: Attendance Classification

def attendance_status(attendance):

    if attendance >= 90:
        return "Excellent"

    elif attendance >= 75:
        return "Good"

    else:
        return "Average"


df["Attendance_Status"] = df["Attendance_%"].apply(
    attendance_status
)

# STEP 10: Individual Student Report

def individual_report():

    student_id = input("Enter Student ID: ").strip()

    student = df[
        df["Student_ID"].str.upper()
        == student_id.upper()
    ]

    if student.empty:
        print("Student not found!")
        return

    s = student.iloc[0]

    print("\n======================================")
    print("       STUDENT PERFORMANCE REPORT")
    print("======================================")

    print("Student ID       :", s["Student_ID"])
    print("Student Name     :", s["Student_Name"])

    print("\nATTENDANCE")
    print("--------------------------------------")

    print("Attendance       :", str(s["Attendance_%"]) + "%")
    print("Attendance Status:", s["Attendance_Status"])

    print("\nACADEMIC PERFORMANCE")
    print("--------------------------------------")

    for subject in subjects:

        print(
            subject.replace("_Marks", ""),
            ":",
            str(s[subject]) + "/100"
        )

    print("--------------------------------------")

    print("Total Marks      :", str(s["Total_Marks"]) + "/300")
    print("Percentage       :", str(s["Percentage"]) + "%")
    print("Best Subject     :", s["Best_Subject"])
    print("Needs Improvement:", s["Needs_Improvement"])
    print("Class Rank       :", s["Class_Rank"])

    print("======================================")


# STEP 11: Class Ranking

def class_ranking():

    ranking = df.sort_values(
        by=["Total_Marks", "Student_ID"],
        ascending=[False, True]
    )

    print("\n==============================================")
    print("                 CLASS RANKING")
    print("==============================================")

    print(
        ranking[
            ["Class_Rank", "Student_ID", "Student_Name",
             "Total_Marks", "Percentage"]
        ].to_string(index=False)
    )

    print("==============================================")


# STEP 12: Attendance Report

def attendance_report():

    print("\n==============================================")
    print("                ATTENDANCE REPORT")
    print("==============================================")

    print(
        df[
            ["Student_ID", "Student_Name",
             "Attendance_%", "Attendance_Status"]
        ].to_string(index=False)
    )

    print("\nATTENDANCE SUMMARY")
    print("----------------------------------------------")

    print(
        "Excellent:",
        (df["Attendance_Status"] == "Excellent").sum()
    )

    print(
        "Good:",
        (df["Attendance_Status"] == "Good").sum()
    )

    print(
        "Average:",
        (df["Attendance_Status"] == "Average").sum()
    )

    print("==============================================")


# STEP 13: Academic Performance Analysis

def academic_analysis():

    print("\n==============================================")
    print("       ACADEMIC PERFORMANCE ANALYSIS")
    print("==============================================")

    print("\nSUBJECT-WISE CLASS AVERAGE")

    for subject in subjects:

        average = df[subject].mean()

        print(
            subject.replace("_Marks", ""),
            ":",
            round(average, 2),
            "/100"
        )

    print("\nTOPPER")

    highest_marks = df["Total_Marks"].max()

    toppers = df[df["Total_Marks"] == highest_marks]

    for _, topper in toppers.iterrows():

        print("Name:", topper["Student_Name"])
        print("Total Marks:", topper["Total_Marks"], "/300")
        print("Percentage:", topper["Percentage"], "%")

    print(
        "\nCLASS AVERAGE PERCENTAGE:",
        round(df["Percentage"].mean(), 2),
        "%"
    )

    print("Total Students:", len(df))

    print("==============================================")


# STEP 14: Export Complete Excel Report

def export_report():

    output_file = "Student_Performance_Attendance_Report.xlsx"

    ranking = df.sort_values(
        by=["Total_Marks", "Student_ID"],
        ascending=[False, True]
    )

    attendance_data = df[
        ["Student_ID", "Student_Name",
         "Attendance_%", "Attendance_Status"]
    ]

    subject_averages = pd.DataFrame({
        "Subject": [s.replace("_Marks", "") for s in subjects],
        "Class_Average": [round(df[s].mean(), 2) for s in subjects]
    })

    attendance_summary = pd.DataFrame({
        "Attendance_Status": ["Excellent", "Good", "Average"],
        "Total_Students": [
            (df["Attendance_Status"] == "Excellent").sum(),
            (df["Attendance_Status"] == "Good").sum(),
            (df["Attendance_Status"] == "Average").sum()
        ]
    })

    topper = df.loc[df["Total_Marks"].idxmax()]

    academic_summary = pd.DataFrame({
        "Metric": [
            "Total Students",
            "Class Average Percentage",
            "Highest Total Marks",
            "Topper Name"
        ],
        "Value": [
            len(df),
            round(df["Percentage"].mean(), 2),
            topper["Total_Marks"],
            topper["Student_Name"]
        ]
    })

    with pd.ExcelWriter(output_file) as writer:

        df.to_excel(
            writer,
            sheet_name="Student_Data",
            index=False
        )

        ranking.to_excel(
            writer,
            sheet_name="Class_Ranking",
            index=False
        )

        attendance_data.to_excel(
            writer,
            sheet_name="Attendance_Report",
            index=False
        )

        attendance_summary.to_excel(
            writer,
            sheet_name="Attendance_Summary",
            index=False
        )

        subject_averages.to_excel(
            writer,
            sheet_name="Subject_Averages",
            index=False
        )

        academic_summary.to_excel(
            writer,
            sheet_name="Academic_Summary",
            index=False
        )

    print("\nExcel report generated successfully!")

    files.download(output_file)


# STEP 15: Main Menu

while True:

    print("\n")
    print("==============================================")
    print("       STUDENT MANAGEMENT SYSTEM")
    print("==============================================")

    print("1. View Individual Student Report")
    print("2. View Class Ranking")
    print("3. View Attendance Report")
    print("4. View Academic Performance Analysis")
    print("5. Export Complete Excel Report")
    print("6. Exit")

    choice = input("\nEnter your choice: ").strip()

    if choice == "1":
        individual_report()

    elif choice == "2":
        class_ranking()

    elif choice == "3":
        attendance_report()

    elif choice == "4":
        academic_analysis()

    elif choice == "5":
        export_report()

    elif choice == "6":

        print("Thank you for using Student Management System!")
        break

    else:

        print("Invalid choice! Please enter 1 to 6.")


Upload Student Performance Dataset


Saving Student_Performance_Attendance_2000_Students.csv to Student_Performance_Attendance_2000_Students.csv


       STUDENT MANAGEMENT SYSTEM
1. View Individual Student Report
2. View Class Ranking
3. View Attendance Report
4. View Academic Performance Analysis
5. Export Complete Excel Report
6. Exit

       STUDENT PERFORMANCE REPORT
Student ID       : STU0034
Student Name     : Tara Bose

ATTENDANCE
--------------------------------------
Attendance       : 66%
Attendance Status: Average

ACADEMIC PERFORMANCE
--------------------------------------
Mathematics : 47/100
Physics : 90/100
Chemistry : 48/100
--------------------------------------
Total Marks      : 185/300
Percentage       : 61.67%
Best Subject     : Physics
Needs Improvement: Mathematics
Class Rank       : 1083


       STUDENT MANAGEMENT SYSTEM
1. View Individual Student Report
2. View Class Ranking
3. View Attendance Report
4. View Academic Performance Analysis
5. Export Complete Excel Report
6. Exit

Enter your choice